# Prompt generation: Causal discovery with logical reasoning

A walk-through for CausalARC prompt generation.

Jacqueline Maasch | August 2025

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns
import json
import platform
import ast
from itertools import permutations,product
from ast import literal_eval
import os
from os import listdir
from os.path import isfile, join

# Custom modules.
os.chdir("../causal_arc")
from carc import CausalARC
from carc_utils import UtilsARC
from carc_augment import AugmentARC
from carc_tasks_logical import TaskLogical
from carc_tasks_extension import TaskExtension
from carc_tasks_order import TaskOrder
from carc_tasks_counting import TaskCounting
from carc_tasks_sprites import TaskSprites

# View versioning.
print("python version     :", platform.python_version())
print("numpy version      :", np.__version__)
print("pandas version     :", pd.__version__)
print("matplotlib version :", matplotlib.__version__)
print("seaborn version    :", sns.__version__)

python version     : 3.12.2
numpy version      : 1.26.4
pandas version     : 2.2.3
matplotlib version : 3.10.0
seaborn version    : 0.13.2


In [2]:
c = CausalARC()
u = UtilsARC()
a = AugmentARC()
tl = TaskLogical()
te = TaskExtension()
to = TaskOrder()
tc = TaskCounting()
ts = TaskSprites()
all_tasks_dict = dict()
all_samples_dict = dict()

# Define functions

In [3]:
def get_prompt_replicates(sample_dict: dict,
                          n_prompts: int = 6,
                          n_examples_in_context: int = 4) -> dict:

    # Get prompt replicates.
    l1_dict = dict()
    l3_dict = dict()
    for i in range(n_prompts):
        
        # Get L1 prompt.
        l1_prompt_dict = c.get_prompt(sample_dict, 
                                      n_examples = n_examples_in_context,
                                      counterfactuals = False,
                                      scm = False,
                                      n_counterfactuals = 0, 
                                      problem_type = "discovery")
        l1_dict[f"Replicate {i}"] = l1_prompt_dict

        # Get L3 prompt.
        l3_prompt_dict = c.get_prompt(sample_dict, 
                                      n_examples = n_examples_in_context//2,
                                      counterfactuals = True,
                                      scm = False,
                                      n_counterfactuals = 1, 
                                      problem_type = "discovery")
        l3_dict[f"Replicate {i}"] = l3_prompt_dict

    return {"L1": l1_dict, "L3": l3_dict}

# Get prompts

In [4]:
n_examples = 10
n_examples_in_context = 4
sizes = [(10,10), (15,15), (20,20)]
colors_list = [[4,5,2,1], [3,6,9,2], [7,8,1,6]]
n_prompts = 5
fun_0 = "and"
fun_1 = "xor"
logical_task_dict = dict()
ttt_sample_dict = dict()
scm = "SCMtcbq"

task_names = []
for size,colors in zip(sizes,colors_list):

    task_name = f"{scm}!{fun_0}!{fun_1}!{size[0]}x{size[1]}"
    task_names.append(task_name)
    print(f"\n-*- {task_name} -*-")

    # Get sample dictionary.
    sample_dict = tl.task_SCMtcbq(fun_0 = fun_0,
                                  fun_1 = fun_1, 
                                 upper_color = colors[0],
                                  middle_color = colors[1],
                                 lower_color = colors[2],
                                 output_color = colors[3],
                                 size = size,
                                 n_examples = n_examples, # Total input-output pairs per sample.
                                 plot = False,
                                 plot_type = "input_output", # "single"
                                 figsize = (5,2),
                                 grid = True)
    ttt_sample_dict[task_name] = sample_dict

    # Get prompt replicates.
    prompt_replicates = get_prompt_replicates(sample_dict,
                                              n_prompts = n_prompts,
                                              n_examples_in_context = n_examples_in_context)
    all_tasks_dict[task_name] = prompt_replicates

    print("\n\nL1 prompt")
    print(prompt_replicates["L1"]["Replicate 0"])

    print("\n\nL3 prompt")
    print(prompt_replicates["L3"]["Replicate 0"])

    #print("\n\nsample_dict")
    #display(sample_dict)

print("\nTotal tasks:", len(task_names))


-*- SCMtcbq!and!xor!10x10 -*-


L1 prompt
You must solve the following causal discovery problem, where the cells in an input array are causal parents of cells in an output array. Both the inputs and outputs are 2D Python arrays of colored pixels. We provide example input-output pairs as demonstration. You must predict the causal function(s) that relate parent cells in the input to their children in the output. Be concise: do not explain your reasoning, and start your answer with 'The logical operators are'.
Example input-output arrays:
[[4, 0, 4, 0, 0, 4, 4, 4, 4, 0], [4, 4, 0, 4, 4, 4, 0, 4, 0, 0], [0, 0, 4, 0, 0, 0, 4, 4, 4, 0], [4, 4, 4, 0, 4, 4, 0, 0, 0, 0], [4, 4, 0, 0, 4, 0, 0, 4, 4, 0], [0, 0, 4, 4, 4, 0, 0, 4, 4, 4], [0, 4, 4, 0, 4, 4, 4, 4, 4, 0], [4, 0, 4, 4, 0, 0, 0, 4, 0, 0], [4, 4, 4, 4, 0, 4, 0, 4, 0, 4], [4, 4, 0, 4, 4, 0, 0, 0, 0, 4], [0, 5, 0, 5, 0, 0, 5, 5, 5, 0], [0, 0, 5, 0, 5, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [5, 0, 0, 5, 5, 5, 5, 0, 5, 5], [5, 0, 0

# Export full JSON

In [5]:
with open(f"../data/causal_discovery_logical_compose/discovery_logical_compose_{fun_0}_{fun_1}.json", "w") as f:
    json.dump(all_tasks_dict, f, indent = 4) # indent for readability.

In [6]:
with open(f"../data/causal_discovery_logical_compose/discovery_logical_compose__{fun_0}_{fun_1}_raw_samples.json", "w") as f:
    json.dump(all_samples_dict, f, indent = 4) # indent for readability.

In [7]:
print("Total tasks:", len(all_tasks_dict.keys()))

Total tasks: 3


In [8]:
all_tasks_dict

{'SCMtcbq!and!xor!10x10': {'L1': {'Replicate 0': "You must solve the following causal discovery problem, where the cells in an input array are causal parents of cells in an output array. Both the inputs and outputs are 2D Python arrays of colored pixels. We provide example input-output pairs as demonstration. You must predict the causal function(s) that relate parent cells in the input to their children in the output. Be concise: do not explain your reasoning, and start your answer with 'The logical operators are'.\nExample input-output arrays:\n[[4, 0, 4, 0, 0, 4, 4, 4, 4, 0], [4, 4, 0, 4, 4, 4, 0, 4, 0, 0], [0, 0, 4, 0, 0, 0, 4, 4, 4, 0], [4, 4, 4, 0, 4, 4, 0, 0, 0, 0], [4, 4, 0, 0, 4, 0, 0, 4, 4, 0], [0, 0, 4, 4, 4, 0, 0, 4, 4, 4], [0, 4, 4, 0, 4, 4, 4, 4, 4, 0], [4, 0, 4, 4, 0, 0, 0, 4, 0, 0], [4, 4, 4, 4, 0, 4, 0, 4, 0, 4], [4, 4, 0, 4, 4, 0, 0, 0, 0, 4], [0, 5, 0, 5, 0, 0, 5, 5, 5, 0], [0, 0, 5, 0, 5, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [5, 0, 0, 5, 5, 5, 5, 0, 5, 5],

# End of document